In [ ]:
# Configuração
PFX_PATH   = "/home/projects/impedidos/Api_Impedidos/certificates/e-CNPJ_F12.p12"
SENHA_PFX  = ""  # senha do PFX
TOKEN_FILE = "/home/projects/impedidos/Api_Impedidos/certificates/token_gov.json"

AUTH_URL  = "https://auth-sigap-rec.ni.estaleiro.serpro.gov.br/recepcao-autenticacao/token"
SIGAP_URL = "https://sigap-impedidos.fazenda.gov.br/impedimento/v2/condicao/{cpf}"

In [ ]:
import json, os, tempfile
from datetime import datetime, timedelta
import requests
from cryptography.hazmat.primitives.serialization import pkcs12, Encoding, PrivateFormat, NoEncryption
from cryptography.hazmat.backends import default_backend


def _extrair_pem(pfx_path, senha):
    with open(pfx_path, "rb") as f:
        pfx_data = f.read()
    private_key, cert, _ = pkcs12.load_key_and_certificates(
        pfx_data, senha.encode(), backend=default_backend()
    )
    tmp_key  = tempfile.NamedTemporaryFile(delete=False, suffix=".key")
    tmp_cert = tempfile.NamedTemporaryFile(delete=False, suffix=".crt")
    tmp_key.write(private_key.private_bytes(Encoding.PEM, PrivateFormat.TraditionalOpenSSL, NoEncryption()))
    tmp_cert.write(cert.public_bytes(Encoding.PEM))
    tmp_key.close(); tmp_cert.close()
    return tmp_cert.name, tmp_key.name


def obter_token():
    # tenta carregar do cache
    if os.path.exists(TOKEN_FILE):
        try:
            with open(TOKEN_FILE) as f:
                cache = json.load(f)
            expira_em = datetime.fromisoformat(cache["expira_em"])
            if datetime.now() < expira_em - timedelta(seconds=300):
                print(f"Token do cache — expira em {expira_em.strftime('%d/%m/%Y %H:%M')}")
                return cache["token"]
        except Exception:
            pass

    # obtém novo via PFX
    print("Obtendo novo token via PFX...")
    cert_path, key_path = _extrair_pem(PFX_PATH, SENHA_PFX)
    try:
        resp = requests.post(AUTH_URL, cert=(cert_path, key_path), verify=True, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        token = data.get("token") or data.get("access_token")
        expires_in = int(data.get("expires_in", 604800))
        expira_em = datetime.now() + timedelta(seconds=expires_in)
        os.makedirs(os.path.dirname(os.path.abspath(TOKEN_FILE)), exist_ok=True)
        with open(TOKEN_FILE, "w") as f:
            json.dump({"token": token, "expira_em": expira_em.isoformat()}, f)
        print(f"Novo token obtido — expira em {expira_em.strftime('%d/%m/%Y %H:%M')}")
        return token
    finally:
        os.unlink(cert_path); os.unlink(key_path)


TOKEN = obter_token()

In [ ]:
# ============================================================
# CONSULTAR CPF
# ============================================================
CPF = "00000000000"   # <- altere aqui

cpf = CPF.replace(".", "").replace("-", "").strip()
headers = {"Authorization": f"Bearer {TOKEN}"}
resp = requests.get(SIGAP_URL.format(cpf=cpf), headers=headers, timeout=15)

if resp.status_code == 200:
    d = resp.json()
    status           = d.get("resultado", "INDEFINIDO")
    motivos          = d.get("motivos", [])
    data_autoexclusao = d.get("dataSolicitacaoAutoexclusao")

    print(f"CPF:              {cpf}")
    print(f"Status:           {status}")
    print(f"Motivos:          {', '.join(motivos) if motivos else '—'}")
    print(f"Data autoexclusão:{' ' + data_autoexclusao if data_autoexclusao else ' —'}")
elif resp.status_code == 404:
    print(f"CPF {cpf} — não encontrado na base")
else:
    print(f"Erro {resp.status_code}: {resp.text}")